# TP2 - Prévision de la consommation électrique avec Stacked LSTM
**Rapport associé :** `Rapport_Stacked_LSTM.pdf` - ES-SAOUDY Zakaria

Reproduction stricte : données minute par minute, 100 000 observations récentes, 10 features,
fenêtres de 60, split 80/20, 3 couches LSTM de 128 unités et 10 epochs.

## 1. Introduction et cadre théorique
Les RNN classiques souffrent du vanishing gradient. Les portes d'oubli, d'entrée et de sortie du LSTM
contrôlent une cellule mémoire. L'empilement hiérarchise les représentations : fluctuations locales,
corrélations sur 1-2 heures, puis cycles globaux. Le dropout 0.3 régularise les couches.

In [ ]:
%pip install -q torch pandas scikit-learn matplotlib

In [ ]:
import urllib.request, zipfile, warnings
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import torch, torch.nn as nn
warnings.filterwarnings('ignore'); torch.manual_seed(42); np.random.seed(42)

## 2. Dataset UCI Household Electric Power Consumption
2 049 280 observations valides après nettoyage, du 16/12/2006 au 26/11/2010. Les valeurs manquantes
sont marquées `?`. La cible est `Global_active_power` (kW).

In [ ]:
DATA = Path('data/stacked_lstm'); DATA.mkdir(parents=True, exist_ok=True)
source = DATA / 'household_power_consumption.txt'
if not source.exists():
    archive = DATA / 'household_power_consumption.zip'
    url = 'https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip'
    urllib.request.urlretrieve(url, archive)
    with zipfile.ZipFile(archive) as bundle: bundle.extractall(DATA)
    source = list(DATA.rglob('household_power_consumption.txt'))[0]

data = pd.read_csv(source, sep=';', na_values=['?'], low_memory=False)
data['datetime'] = pd.to_datetime(data['Date'] + ' ' + data['Time'], format='%d/%m/%Y %H:%M:%S')
data = data.drop(['Date', 'Time'], axis=1).dropna().reset_index(drop=True)
data['hour_sin'] = np.sin(2*np.pi*data['datetime'].dt.hour/24)
data['hour_cos'] = np.cos(2*np.pi*data['datetime'].dt.hour/24)
data['is_weekend'] = (data['datetime'].dt.dayofweek >= 5).astype(int)
print(f'Dataset chargé : {len(data):,} lignes | {data.datetime.min().date()} -> {data.datetime.max().date()}')

## 3. Features
Puissance active (cible), puissance réactive, tension, intensité, trois sous-compteurs, `hour_sin`,
`hour_cos` et `is_weekend`. L'encodage cyclique évite la rupture artificielle entre 23 h et 0 h.

## 4. Normalisation globale et séquences de 60 minutes - pipeline exact du rapport

In [ ]:
cols = ['Global_active_power','Global_reactive_power','Voltage','Global_intensity',
        'Sub_metering_1','Sub_metering_2','Sub_metering_3','hour_sin','hour_cos','is_weekend']
dataset = data[cols].values
scaler = MinMaxScaler(); dataset_scaled = scaler.fit_transform(dataset)
target_scaler = MinMaxScaler(); target_scaler.fit(data[['Global_active_power']])
dataset_scaled = dataset_scaled[-100000:]

def create_sequences(values, seq_length=60):
    X, y = [], []
    for i in range(len(values) - seq_length):
        X.append(values[i:i+seq_length]); y.append(values[i+seq_length, 0])
    return np.array(X), np.array(y)

X, y = create_sequences(dataset_scaled, 60)
split = int(len(X) * .8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
X_train = torch.from_numpy(X_train).float(); X_test = torch.from_numpy(X_test).float()
y_train = torch.from_numpy(y_train).float().unsqueeze(-1)
y_test = torch.from_numpy(y_test).float().unsqueeze(-1)
print('X :', X.shape, '| train :', len(X_train), '| test :', len(X_test))
assert X.shape == (99940, 60, 10) and len(X_train) == 79952 and len(X_test) == 19988

## 5. Architecture : 3 LSTM x 128, dropout 0.3, Linear 128 -> 1

In [ ]:
class StackedLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        return self.fc(lstm_out[:, -1, :])

model = StackedLSTM(input_size=10, hidden_size=128, num_layers=3, dropout=0.3)
parameter_count = sum(p.numel() for p in model.parameters())
print(model, '\nParamètres :', parameter_count)
assert parameter_count == 334465

## 6. Entraînement exact : MSE, Adam 0.001, batch 32, 10 epochs

In [ ]:
criterion = nn.MSELoss(); optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
epoch_losses = []
for epoch in range(10):
    model.train()
    for i in range(0, len(X_train), 32):
        optimizer.zero_grad()
        loss = criterion(model(X_train[i:i+32]), y_train[i:i+32])
        loss.backward(); optimizer.step()
    epoch_losses.append(loss.item())
    print(f'Epoch [{epoch+1}/10] - Loss: {loss.item():.6f}')
plt.plot(range(1,11), epoch_losses, marker='o'); plt.xlabel('Epoch'); plt.ylabel('Loss finale'); plt.grid(); plt.show()

## 7. Prédictions, retour à l'échelle physique et métriques

In [ ]:
model.eval()
with torch.no_grad(): y_pred_scaled = model(X_test)
y_real = target_scaler.inverse_transform(y_test.numpy().reshape(-1,1))
y_pred = target_scaler.inverse_transform(y_pred_scaled.numpy().reshape(-1,1))
mae = mean_absolute_error(y_real, y_pred)
rmse = np.sqrt(mean_squared_error(y_real, y_pred))
print(f'MAE : {mae:.4f} kW'); print(f'RMSE : {rmse:.4f} kW'); print(f'RMSE/MAE : {rmse/mae:.3f}')
plt.figure(figsize=(14,4)); plt.plot(y_real[:500], label='Consommation réelle')
plt.plot(y_pred[:500], label='Prédiction Stacked LSTM', color='red')
plt.legend(); plt.ylabel('kW'); plt.tight_layout(); plt.show()

## 8. Résultats et discussion du rapport
Le rapport obtient MAE `0.1912 kW`, RMSE `0.2898 kW` et ratio `1.516`. Les tendances sont bien suivies,
mais les pics courts sont atténués. Les références données sont LSTM simple ~0.28/~0.40 et persistance
~0.35/~0.55 (valeurs estimées dans le rapport).

## 9. Limites et pistes d'amélioration
- 10 epochs -> 50 epochs + `ReduceLROnPlateau`.
- Instabilité epochs 3-5 -> lr `0.0005` ou warm-up.
- One-step -> seq-to-seq à +5 et +30 minutes.
- Pas de validation -> early stopping sur `val_loss`.
- Pics sous-estimés -> MAE ou Huber.
- Perspectives : mécanisme d'attention et optimisation des hyperparamètres.